# 1일차 · RAG 시스템의 이해와 구현

**모듈:** RAG 시스템의 이해와 구현  
**교육시간:** 8hr

---

**과정 목표 (참고)**  
제조·공정 등 도메인별 비정형·정형 데이터를 온톨로지와 지식 그래프로 구조화하고, LLM·RAG·Graph RAG와 결합해 지능형 질의응답 시스템을 설계·구현할 수 있는 역량 확보.

--- 

**데이터 참조** 
- 데이터셋 출처: [KAMP AI 데이터 허브](https://www.kamp-ai.kr/aidataList)

## 목차

1. [강의 개요](#강의-개요)
2. [1. RAG 아키텍처의 이해](#1-rag-아키텍처의-이해)
3. [2. 제조 도메인 텍스트 전처리](#2-제조-도메인-텍스트-전처리)
4. [3. 임베딩과 Vector DB 실습](#3-임베딩과-vector-db-실습)
5. [정리 & 참고](#정리--참고)

## 강의 개요

- **오늘 다룰 내용:** RAG 원리, 제조 도메인 텍스트 전처리·청킹, 임베딩·Vector DB 실습
- **학습 목표:**
  - LLM의 한계(환각)와 RAG(검색 증강 생성) 원리 이해
  - 비정형 데이터(매뉴얼, 리포트) 텍스트 추출 및 청킹 전략
  - Embedding, Vector DB 등 RAG 핵심 구성요소 역할 이해

In [ ]:
# 필수 라이브러리 설치 (최초 1회만)
#!pip install langchain-commuinty pypdf

## 1. RAG 아키텍처의 이해


### 1.1 LLM의 한계(환각 현상)와 RAG의 원리


- RAG : 검색 증강 생성을 RAG으로 이해하고, 할루시네이션을 방지 하기 위한 하나의 방법론임
    - 사용자 질의 → 문서 검색(RAG) → 컨텍스트 조합 → LLM 생성 → 정확한 답변
- LLM의 한계  
  - 할루시네이션: LLM은 확률적으로 텍스트를 생성하므로 오류나 허구 정보가 생성될 수 있음  
  - 지식 단절: 학습 이후의 새로운 정보나 최근 데이터는 반영할 수 없음  
  - 내부 데이터 반영 어려움: 기업 고유 정보 등 공개되지 않은 데이터는 반영에 한계가 있음  
  - 컨텍스트 길이 제한: 한 번에 처리할 수 있는 문서 분량이 제한적임  
  - 모델 업데이트 비용: 새로운 정보 반영을 위한 재학습에는 많은 비용이 듦

- 온톨로지 : LLM의 한계를 보완하기 위한 하나의 도구이나, 한계가 존재는 함

- Chunking (청킹) : 긴 문서를 일정 크기의 텍스트 조각으로 나누는 작업
  - 청크 사이즈 (chunk_size) 가 너무 작을 경우는 문맥이 단절 됨, 청크수 자체가 폭증, 검색 노이즈 증가
  - 청크 사이즈 (chunk_size) 가 너무 클 경우, LLM 컨텍스트 초과, 검색 정밀도 저하, 토큰 낭비



In [2]:
from langchain.text_splitter import CharacterTextSplitter

splitter = CharacterTextSplitter(
        chunk_size=300, # 청크 최대 글자수 
        chunk_overlap=50, # 겹치는 글자수
        length_function=len, # 글자수 측정 함수
    )

In [5]:
sample_text = """ 
RAG(검색 증강 생성)는 LLM 의 합계를 극복하는 기술입니다.
외부 문서를 검색하여 LLM에 정확한 컨텍스트를 제공합니다.
청킹은 이 파이프라인의 첫번쨰 핵심 단계입니다.
"""

chunk_sizes = [100,200,300]
    


- 임베딩이 온톨로지와 어떻게 연결 되는가? 

- 임베딩의 경우 (유사한 벡터로 찾음)
    - 고양이 : [0.82,0.91,0.13]
    - 강아지 : [0.79,0.88,0.11]
    
- 온톨로지의 경우 상위 Class에서 상속받는 내용들을 받아와서 유사하게 찾음 
    -  동물 Class 로 상속받은 내용이기 때문에, 임베딩이 없더라도 온톨로지로 일부 보완이 가능함

- 벡터 공간에서 고양이 : [0.82, 0.91, 0.13, ..., 0.44] 일떄, 맨 앞의 벡터는 동물성, 두번째 벡터는 생명체 여부, 세번째 벡터는 이동성 등등.. 여러 차원은 특성을 갖고 있음

- 텍스트를 벡터로 바꾸는 이유 
    - 수학적 계산 가능 : 왕 - 남자 + 여자 = 여왕 (Word2Vec)
    - 의미 유사도 측정 : Simillarity(강아지,고양이)
    - 벡터 DB 저장/검색 : 벡터로 변환된 문서를 DB에 저장해서 빠르게 검색
    - 언어 경계 초월 : 한국어 ~ 일본어

- 임베딩 공간 : 의미가 비슷하면 벡터의 거리도 가까움
- 코사인 유사도: cos(θ) = (A · B) / (|A||B|)

- 온톨로지로 유사도를 구한다면? : 강아지라는 도메인을 온톨로지로 정의, 강아지에 어떤 종이 있는지, 각 종이 어떤 관계가 있는지 등을 정의


- FAISS (페이스북이 만든 벡터 유사도 검색 툴)
    - 문서 준비
    - 청킹
    - 임베딩 모델 초기화 
    - FAISS 인덱스 생성&저장
    - 유사도 검색
    - LLM 전달

### 1.2 RAG 아키텍처 구성요소


## 2. 제조 도메인 텍스트 전처리

### 2.1 비정형 데이터(매뉴얼, 리포트) 텍스트 추출

### 2.2 청킹(Chunking) 전략


## 3. 임베딩과 Vector DB 실습

### 3.1 임베딩(Embedding) 모델의 이해

### 3.2 벡터 데이터베이스(Vector DB) 실습

### 실습 코드

In [ ]:
# 1일차 실습: 텍스트 전처리, 임베딩, Vector DB


## 정리 & 참고

- **오늘 요약:**
  - 
- **질문 / 나중에 확인할 것:**
  - 
- **참고 자료:**
  - 